#### Middleware : Provides a way to tightly control whats happenign inside the agent. Middleware is useful for the following 
 - Trackign agent behaviour with logging, analytics and debugging
 - Transforming prompts, tool selection and o/p formatting
 - Adding retries, fallbacks and early termination logic.
 - Applying rate limits, gaurdrails and PII detection.
 

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

#### Summarization Middleware
- Automatically summarizes conversation history whne approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following :
- Long-running conversations that exceed context windows.
- Multi-run dailogues with excessive history
- Applications where preserving full conversation context matters.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage


In [ ]:
## Message based summarization
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages",10), ## summarization trigger based on message length
            keep=("messages",4)
        )
    ]
)

In [4]:
## Run with thead ID
config={"configurable":{"thread_id":"test-1"}}

In [8]:
# Alternative test data
questions = [
    "what is 2+2 ?",
    "what is 10*5 ?",
    "what is 1000/24 ?",
    "what is 1000+24 ?",
    "what is 10-24 ?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages:{response}")
    print(f"Messages:{len(response['messages'])}")

Messages:{'messages': [HumanMessage(content='what is 2+2 ?', additional_kwargs={}, response_metadata={}, id='5ca6709a-bc05-423a-8c44-e41930a981ef'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 14, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_944bbf963c', 'id': 'chatcmpl-Dnt4WBZDTDYB3AOgA6snR107XhxzP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e9ee3-0854-7392-b2a1-fb7206113bb0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 8, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

In [10]:
## trigger basee on token size

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage

@tool
def search_hotels(city:str) -> str:
    """ search hotels - return long response to use more tokens """
    return f"""hotels in {city}:
    1. Grand Hotel - 5 star , 530$/night , spa, pool gym
    2. City Inn - 4 star, 180$/night, Business center
    3. Budget Stay - 3 star , 75$/night, free wifi """

agent = create_agent(
    model="gpt-4o-mini",
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens",550), ## summarization trigger based on number of tokens
            keep=("tokens",200)
        ),
    ]
)
config={"configurable":{"thread_id":"test-1"}}

## token counter (approximate)

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 ## 4 chars = 1 token 


In [11]:
## Run test 

cities = ["Paris", "London", "Dubai", "Hyderabad","Singapore", "New york"]

for city in cities:
    response  = agent.invoke(
        {"messages":[HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )
tokens=count_tokens(response['messages'])
print(f"{city}: ~{tokens} tokens,{len(response['messages'])} messages")
print (f"{(response['messages'])}")


New york: ~435 tokens,9 messages
[HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to find hotels in Paris, London, Dubai, and Hyderabad.\n\n## SUMMARY\nThe user requested hotels in:\n- **Paris:** Options provided were:\n  1. Grand Hotel - 5 stars, $530/night, amenities: spa, pool, gym.\n  2. City Inn - 4 stars, $180/night, amenities: business center.\n  3. Budget Stay - 3 stars, $75/night, amenities: free WiFi.\n\n- **London:** The same list of hotels as in Paris was provided.\n\n- **Dubai:** The same three hotel options were provided as before.\n\n- **Hyderabad:** The same three hotel options were provided:\n  1. Grand Hotel - 5 stars, $530/night, amenities: spa, pool, gym.\n  2. City Inn - 4 stars, $180/night, amenities: business center.\n  3. Budget Stay - 3 stars, $75/night, amenities: free WiFi.\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nNone.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={

#### Human loop in middleware :  
Pause agent execution for human intervention of tool calls before they execute. Human in lopp is useful for following:
- High stakes operations requiring human approval (e.g database writes, financial transactions)
- Compliance workflows where human oversight is mandatory
- Long-running converesations where human feedabck guides the agent

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """mock function to read an email by its ID"""
    return f"Email content for ID {email_id}"

def send_email_tool(recipient: str, subject: str, body : str) -> str:
    """Mock function to send email"""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="gpt-4o",
    tools = [read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
             interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,
             }
        ),
    ]
)

In [31]:
config={"configurable":{"thread_id":"test-approve"}}
# step 1 : request
result = agent.invoke(
    {"messages":[HumanMessage(content=f"send email to Rao@test.com with subject 'Hello' and body 'How are you?' ")]},
        config=config
)

In [32]:
result

{'messages': [HumanMessage(content="send email to Rao@test.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='9fe6eca2-21b4-4780-8b88-aafb990ef721'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 97, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_5f78e76dfa', 'id': 'chatcmpl-DoBJ73cQi2lg60jJFzL0OcclSQYrB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ea310-a51a-71c3-8854-95969951f91b-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'Rao@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_sFCYSkf2pkDW2if9Q

In [33]:
# Step 2 Approve
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
#config={"configurable":{"thread_id":"test-approve"}}
if "__interrupt__" in result:
    print("pause approving....")
    result = agent.invoke(
        Command( 
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
    
    config = config
    )

    print(f"Result: {result['messages'][-1].content}")

pause approving....
Result: The email has been sent to Rao@test.com with the subject "Hello".


In [34]:
result

{'messages': [HumanMessage(content="send email to Rao@test.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='9fe6eca2-21b4-4780-8b88-aafb990ef721'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 97, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_5f78e76dfa', 'id': 'chatcmpl-DoBJ73cQi2lg60jJFzL0OcclSQYrB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ea310-a51a-71c3-8854-95969951f91b-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'Rao@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_sFCYSkf2pkDW2if9Q

In [35]:
## Reject scenario

 
 
config={"configurable":{"thread_id":"test-reject"}}
result = agent.invoke(
    {"messages":[HumanMessage(content=f"send email to Rao@test.com with subject 'Hello' and body 'How are you?' ")]},
        config=config
)

In [36]:
##Reject

 
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

if "__interrupt__" in result:
    print("pause approving....")
    result = agent.invoke(
        Command( 
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
    
    config = config
    )

    print(f"Result: {result['messages'][-1].content}")

pause approving....
Result: It seems there was an issue with sending the email. If you'd like to try again or need further assistance, please let me know!


In [37]:
result

{'messages': [HumanMessage(content="send email to Rao@test.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='ea4e8485-dc7d-4588-b0ef-1dfa1d6f07a9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 97, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_5f78e76dfa', 'id': 'chatcmpl-DoCAa1nfecYvSK6TpRv3WLynnaydZ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ea343-363a-7603-aa0b-713c1dc8016a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'Rao@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': 'call_vw4zlXug1TJI4BbBw